# Step 3: Extract and Chunk Text-Layer Documents (Google Colab)
### Project: Multilingual RAG Chatbot on Cooperative Law (J&K, Punjab, Haryana, Central)

This notebook automates the extraction, chunking, and quality audit for the **29 documents** that have a selectable text layer.

- **Base directory in Google Drive**: `/content/drive/MyDrive/MDC-project 26`
- **Manifest**: `manifest.csv`
- **Raw PDFs**: `raw/` subfolders (`Central law and PACS/`, `grievance/`, `haryana/`, `jk/`, `kcc_finance/`, `pmfby/`, `punjab/`)
- **Script**: `extract_and_chunk.py`
- **Output directory**: `processed/`

## 1. Install Dependencies
Installs Tesseract OCR (with English, Hindi, Punjabi models) and the required Python libraries.

In [ ]:
# Install Tesseract OCR and Indian language packs
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-pan

# Install Python packages
!pip -q install pymupdf pytesseract pillow pandas tabulate

# Check Tesseract installation
!tesseract --version

## 2. Mount Google Drive and Verify File Structure
Mounts Google Drive and verifies that `raw/`, `manifest.csv`, and `extract_and_chunk.py` are present.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/MDC-project 26'
RAW = f'{BASE}/raw'
MANIFEST = f'{BASE}/manifest.csv'
OUT = f'{BASE}/processed'
SCRIPT = f'{BASE}/extract_and_chunk.py'

print('Verifying Project Paths:')
for name, p in [('BASE FOLDER', BASE), ('RAW FOLDER', RAW), ('MANIFEST', MANIFEST), ('SCRIPT', SCRIPT)]:
    status = 'FOUND' if os.path.exists(p) else 'MISSING'
    print(f'  [{status}] {name:15s}: {p}')

assert os.path.exists(SCRIPT), f'Missing script at {SCRIPT}. Ensure extract_and_chunk.py is uploaded to MDC-project 26.'
assert os.path.exists(MANIFEST), f'Missing manifest at {MANIFEST}. Ensure manifest.csv is in MDC-project 26.'
assert os.path.exists(RAW), f'Missing raw folder at {RAW}. Ensure raw/ is in MDC-project 26.'

## 3. Run extract_and_chunk.py in Text Mode
Runs `extract_and_chunk.py` with `--mode text` for only documents that already have a selectable text layer (29 expected). Stray blank pages are automatically OCR'd.

In [ ]:
!python "$SCRIPT" --raw "$RAW" --manifest "$MANIFEST" --out "$OUT" --mode text

## 4. Quality & Processing Report
Reads `processed/processing_report.csv`:
- Confirms document count (expected: 29)
- Displays chunk counts per document
- Shows every row with a non-empty `flags` field

In [ ]:
import pandas as pd
from IPython.display import display

rep_path = f'{OUT}/processing_report.csv'
df = pd.read_csv(rep_path)

print('========================================')
print(f'Total Documents Processed: {len(df)} (Expected: 29)')
print(f'Total Chunks Created     : {df["chunks"].sum()}')
print('========================================\n')

print('--- Chunk Counts Per Document ---')
display(df[['doc_id', 'state', 'module', 'pages', 'chunks', 'avg_chunk_chars', 'flags']])

flagged = df[df['flags'].notna() & (df['flags'].astype(str).str.strip() != '')]
print(f'\n--- Documents with Flags ({len(flagged)}) ---')
if len(flagged) > 0:
    display(flagged[['doc_id', 'pages', 'chunks', 'flags']])
else:
    print('No flags encountered! All documents processed cleanly.')

## 5 & 6. PyMuPDF Verification Against Real PDF Text
For the 6 key target documents, we compare 3 chunks each with the real PDF page text using PyMuPDF (`fitz`):
1. `mscs_act_2002_amendment_2023`
2. `haryana_co_operative_societies_act_1984_pdf`
3. `act_1989_cooperation_gov_in_copy`
4. `final_pmfby_operational_guidelines_2023_120125`
5. `kcc_master_circular_2018`
6. `2023_rule`

**Auditing Criteria:**
- Does each chunk start at a Section / Rule / Heading?
- Did headers, footers, or page numbers leak in?
- Are sentences cut in the middle?

In [ ]:
import json, re, csv
from pathlib import Path
import pymupdf as fitz
import pandas as pd
from tabulate import tabulate

target_docs = [
    'mscs_act_2002_amendment_2023',
    'haryana_co_operative_societies_act_1984_pdf',
    'act_1989_cooperation_gov_in_copy',
    'final_pmfby_operational_guidelines_2023_120125',
    'kcc_master_circular_2018',
    '2023_rule'
]

# Load manifest map
with open(MANIFEST, newline='', encoding='utf-8-sig') as f:
    manifest_map = {r['doc_id']: r for r in csv.DictReader(f)}

PAGE_NUM_PAT = re.compile(r'^[\-–—\s]*(page\s*)?\d{1,4}(\s*(of|/)\s*\d{1,4})?[\-–—\s]*$', re.I)
SECTION_START_PAT = re.compile(
    r'^(\d{1,3}[A-Z]{0,2}\.|CHAPTER|PART|THE\s+SCHEDULE|ANNEXURE|APPENDIX|' \
    r'\d{1,2}(\.\d{1,2}){0,3}\.?\s+[A-Z]|[A-Z0-9 ,&/()\-]{4,80}$)', re.M)

audit_results = []

print('=' * 80)
print('STEP 5: AUDITING 3 CHUNKS PER TARGET DOCUMENT AGAINST REAL PDF TEXT')
print('=' * 80)

for did in target_docs:
    chunk_file = Path(OUT) / 'chunks' / f'{did}.jsonl'
    if not chunk_file.exists():
        audit_results.append({
            'doc_id': did, 'chunks': 0, 'problem found (Y/N)': 'Y', 'what is wrong': 'Chunk file not found'
        })
        continue

    with open(chunk_file, 'r', encoding='utf-8') as f:
        chunks = [json.loads(line) for line in f if line.strip()]

    if not chunks:
        audit_results.append({
            'doc_id': did, 'chunks': 0, 'problem found (Y/N)': 'Y', 'what is wrong': 'No chunks extracted'
        })
        continue

    # Pick 3 chunks across the document
    sample_indices = [0, len(chunks)//2, len(chunks)-1] if len(chunks) >= 3 else list(range(len(chunks)))
    samples = [chunks[i] for i in sorted(set(sample_indices))]

    # Load raw PDF using PyMuPDF
    local_rel = manifest_map.get(did, {}).get('local_path', '')
    pdf_path = Path(RAW) / local_rel
    pdf_doc = fitz.open(pdf_path) if pdf_path.exists() else None

    issues = []
    print(f'\n>>> DOCUMENT: {did} (Total chunks: {len(chunks)}, PDF: {local_rel})')

    for idx, c in enumerate(samples, 1):
        cid = c.get('chunk_id', f'chunk_{idx}')
        text = c.get('text', '').strip()
        p_start, p_end = c.get('page_start', 1), c.get('page_end', 1)
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        first_l = lines[0] if lines else ''
        last_l = lines[-1] if lines else ''

        # Check 1: starts at section / heading
        starts_sec = bool(SECTION_START_PAT.match(first_l)) or bool(re.match(r'^[A-Z\"\'(]', first_l))
        # Check 2: page number leak
        page_leak = any(PAGE_NUM_PAT.match(l) for l in lines)
        # Check 3: cut sentence
        cut_sent = bool(last_l and last_l[-1] not in '.:;—–-)?"'’”)।')

        # Read real PDF text from PyMuPDF
        pdf_text_sample = ''
        if pdf_doc and 1 <= p_start <= len(pdf_doc):
            pdf_text_sample = pdf_doc[p_start - 1].get_text('text').strip()[:120].replace('\n', ' ')

        print(f'  Sample [{idx}/3] ID: {cid} | Pages {p_start}-{p_end}')
        print(f'    First line : {first_l[:80]}')
        print(f'    Last line  : {last_l[-80:]}')
        if pdf_text_sample:
            print(f'    PDF Page {p_start} : {pdf_text_sample}...')
        print(f'    Starts Section/Heading: {starts_sec} | Page Num Leaked: {page_leak} | Cut Mid-Sentence: {cut_sent}')

        if not starts_sec:
            issues.append(f'{cid}: starts mid-clause/lowercase')
        if page_leak:
            issues.append(f'{cid}: page number line leaked')

    if pdf_doc:
        pdf_doc.close()

    has_problem = 'Y' if issues else 'N'
    desc = '; '.join(issues) if issues else 'None. Clean headings, no page/footer leaks, valid boundaries.'
    audit_results.append({
        'doc_id': did,
        'chunks': len(chunks),
        'problem found (Y/N)': has_problem,
        'what is wrong': desc
    })

# STEP 6: Display Markdown Table
results_df = pd.DataFrame(audit_results)
print('\n' + '=' * 80)
print('STEP 6: FINAL TABLE')
print('=' * 80 + '\n')
print(tabulate(results_df, headers=['doc_id', 'chunks', 'problem found (Y/N)', 'what is wrong'], tablefmt='github', showindex=False))
display(results_df)